In [6]:
import pickle
from pathlib import Path
import os
from tqdm import tqdm
import jsonl

# Extract sound

In [ ]:
# Check cache
if Path("/Volumes/Index/Cache/sound.wav").exists():
    print("The result of this step is cached. Skipping the task")
else:
    print("No cache found. Running the task")

In [ ]:
import subprocess

SOURCE = "/Volumes/Index/BetterCapture/video.mp4"
TARGET = "/Volumes/Index/Cache/sound.wav" # the format must be .wav with the command at hand

res = subprocess.run(
    ['ffmpeg', '-i', SOURCE, '-vn', '-c:a', 'pcm_s16le', TARGET],
    capture_output=True
)
if res.returncode == 1:
    raise ValueError(f"ffmpeg call failed: \n{res.stdout.decode("UTF-8")}")
del res

# Diarize

* Drops utterances that are 1 second long or less

In [ ]:
# Check cache
if Path("/Volumes/Index/Cache/diarization.pkl").exists():
    print("Diarization exists. Skipping the task")
    with open("/Volumes/Index/Cache/diarization.pkl", mode="rb") as file:
        output = pickle.load(file)
else:
    print("No cache found. Running the task")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import pyannote
from pyannote.audio import Pipeline

In [ ]:
# Costly cell. Run only if no cache
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1", 
    token=os.environ["HF_TOKEN"]
)

In [ ]:
# VERY costly cell. Run only if nothing is cached
output = pipeline("/Volumes/Index/Cache/sound.wav")

for turn, speaker in output.speaker_diarization:
    print(f"{speaker} speaks between t={turn.start:.3f}s and t={turn.end:.3f}s")

In [ ]:
output.exclusive_speaker_diarization

In [ ]:
with open("/Volumes/Index/Cache/diarization.pkl", mode="wb") as file:
    pickle.dump(output, file)

In [ ]:
segments = [i for i in output.exclusive_speaker_diarization if i[0].end - i[0].start > 1]
segments[:5]

In [ ]:
json_lines =[{"speaker_id": i[1], "start": i[0].start, "stop": i[0].end} for i in segments]
json_lines[:3]

In [ ]:
import jsonl
jsonl.dump(json_lines, "/Volumes/Index/Cache/meeting-diary.jsonl")

# Slice audio

In [ ]:
import jsonl
diary = list(jsonl.load("/Volumes/Index/Cache/meeting-diary.jsonl"))
diary[:5]

In [ ]:
if not Path("/Volumes/Index/Cache/slices/").exists():
    os.mkdir("/Volumes/Index/Cache/slices")

In [ ]:
import subprocess


for i in tqdm(diary):
    speaker = str(i["speaker_id"])
    start = str(i["start"])
    end = str(i["stop"])
    
    filename = f"{speaker}-{start.replace(".","_")}-{end.replace(".","_")}.wav"
    
    res = subprocess.run(
        ["ffmpeg", "-ss", start, "-to", end, "-i", "/Volumes/Index/Cache/sound.wav", 
         "-c:a", "pcm_s16le", "/Volumes/Index/Cache/slices/" + filename],
        capture_output=True
    )

# STT

In [ ]:
30 % 60

In [ ]:
from datetime import time

def seconds_to_time(string_time:str) -> time:
    if ":" not in string_time:
        raise ValueError("The separator ':' not found in time")

    s, ms = [int(i) for i in string_time.split(":")]
    h = s // 3600
    s = s % 3600
    
    m = s // 60
    s = s % 60

    ms = int(float("0." + str(ms)) * 1000000)
    return time(h, m, s, ms)

In [ ]:
for i in Path("/Volumes/Index/Cache/slices").iterdir():
    if i.name.startswith("."):
        continue
    if not i.name.endswith(".wav"):
        raise ValueError(f"An invalid slice: {i.name}")
print("All are valid!")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from faster_whisper import WhisperModel

transcipt = []

model = WhisperModel("small", compute_type="int8")

dir_size = len(list(Path("/Volumes/Index/Cache/slices").iterdir()))
for path in tqdm(Path("/Volumes/Index/Cache/slices").iterdir(), total=dir_size):
    segments, info = model.transcribe(path.absolute(), language="en")
    
    speaker, start, end = path.name.split("-")

    start = start.replace("_", ":")
    start = seconds_to_time(start)
    
    end = end[:-4].replace("_", ":")
    end = seconds_to_time(end)
    
    utteration = "".join(segment.text for segment in segments).strip()
    
    if not utteration:
        continue
    
    transcipt.append({
        "speaker": speaker, 
        "start": start.strftime("%H:%M:%S.%f"),
        "end": end.strftime("%H:%M:%S.%f"),
        "utteration": utteration
    })

transcipt.sort(key=lambda x: time.fromisoformat(x["start"]))
transcipt[:3]

In [ ]:
with open("/Volumes/Index/Cache/transcipt.pkl", mode="wb") as f:
    pickle.dump(transcipt, f)

# Packaging

In [3]:
with open("/Volumes/Index/Cache/transcipt.pkl", mode="rb") as f:
    transcript = pickle.load(f)
transcript[:3]

[{'speaker': 'SPEAKER_01',
  'start': '00:00:00.309687',
  'end': '00:00:01.650968',
  'utteration': "I don't see that it's so spooky"},
 {'speaker': 'SPEAKER_01',
  'start': '00:00:03.929093',
  'end': '00:00:09.227843',
  'utteration': 'Hi everybody! Welcome back to Sweet and Sour. Today we are on episode...'},
 {'speaker': 'SPEAKER_02',
  'start': '00:00:11.995343',
  'end': '00:00:13.193468',
  'utteration': 'doing a 670 mount.'}]

In [5]:
speaker_mapping = {
    "SPEAKER_00": "Youtube dude",
    "SPEAKER_01": "Poki",
    "SPEAKER_02": "Lily"
}

clean_transcript = []
for i in transcript:
    j = i.copy()
    j["speaker"] = speaker_mapping[j["speaker"]]
    clean_transcript.append(j)
clean_transcript[:3]

[{'speaker': 'Poki',
  'start': '00:00:00.309687',
  'end': '00:00:01.650968',
  'utteration': "I don't see that it's so spooky"},
 {'speaker': 'Poki',
  'start': '00:00:03.929093',
  'end': '00:00:09.227843',
  'utteration': 'Hi everybody! Welcome back to Sweet and Sour. Today we are on episode...'},
 {'speaker': 'Lily',
  'start': '00:00:11.995343',
  'end': '00:00:13.193468',
  'utteration': 'doing a 670 mount.'}]

In [8]:
import jsonl
jsonl.dump(clean_transcript, "/Volumes/Index/Cache/meeting.jsonl")